# Lekcija 6: Svojstveni vektori i svojstvene vrijednosti matrice kovarijance dobivene iz momenata

U Lekciji 5 pomoću središnjih momenata odredili smo orijentaciju područja te, uz malo dodatne algebre, duljine njegovih poluosi. Ova lekcija tu vezu objašnjava izravno i općenito: središnji momenti *bilo kojeg* područja, a ne samo elipse, definiraju **matricu kovarijance** veličine $2\times2$. Svojstveni vektori i svojstvene vrijednosti te matrice izravno daju orijentaciju i veličinu *ekvivalentne elipse* &mdash; elipse s istom površinom i istom raspršenošću prema momentima drugog reda kao promatrano područje.


## Podsjetnik: što su svojstveni vektori i svojstvene vrijednosti?

Za kvadratnu matricu $A$, nenulti vektor $v$ jest **svojstveni vektor** sa **svojstvenom vrijednošću** $\lambda$ ako vrijedi

$$Av = \lambda v$$

Drugim riječima: primjena matrice $A$ na vektor $v$ ne zakreće vektor $v$ izvan pravca na kojem se nalazi &mdash; samo ga *skalira* faktorom $\lambda$. Matrica $A$ većinu vektora i zakreće i skalira; svojstveni vektori predstavljaju posebne smjerove duž kojih dolazi samo do skaliranja.

Za **simetričnu** matricu veličine $2\times2$, poput matrica kovarijance koje ćemo izraditi u nastavku, zajamčeno je još jedno korisno svojstvo: uvijek postoje dva međusobno okomita svojstvena vektora, a njihove svojstvene vrijednosti realni su brojevi. Geometrijski, matrica $A$ preslikava kružnicu jediničnih vektora u elipsu čije osi slijede smjerove svojstvenih vektora, pri čemu je duljina svake poluosi jednaka odgovarajućoj svojstvenoj vrijednosti. Upravo tu predodžbu &mdash; svojstveni vektori kao *smjerovi* osi, a svojstvene vrijednosti kao njihove *duljine* &mdash; koristit ćemo u analizi oblika u nastavku.


In [ ]:
import numpy as np
import cv2
import matplotlib.pyplot as plt

A = np.array([[3.0, 1.0],
              [1.0, 1.5]])

eigvals, eigvecs = np.linalg.eigh(A)  # ascending order
print('eigenvalues :', eigvals)
print('eigenvectors (columns):\n', eigvecs)

theta = np.linspace(0, 2 * np.pi, 200)
circle = np.stack([np.cos(theta), np.sin(theta)])  # unit circle, as column vectors
ellipse = A @ circle  # apply A to every point on the circle

fig, ax = plt.subplots(figsize=(4.5, 4.5))
ax.plot(circle[0], circle[1], '--', color='gray', label='unit circle')
ax.plot(ellipse[0], ellipse[1], color='#3498db', label='A @ circle')
for val, vec in zip(eigvals, eigvecs.T):
    ax.plot([0, val * vec[0]], [0, val * vec[1]], color='red', linewidth=2)
ax.scatter(0, 0, color='black', zorder=5)
ax.set_aspect('equal')
ax.legend(loc='upper left', fontsize=8)
ax.set_title('A circle stretched by A; red lines = eigenvectors x eigenvalues')
plt.show()

### Od jednog svojstvenog vektora do obaju odjednom: dijagonalizacija

Prethodni izraz $Av = \lambda v$ vrijedi za jedan svojstveni vektor. Postavimo *oba* svojstvena vektora kao stupce matrice $P = \begin{bmatrix}v_1 & v_2\end{bmatrix}$, a obje svojstvene vrijednosti na dijagonalu matrice $\Lambda = \begin{bmatrix}\lambda_1 & 0\\0 & \lambda_2\end{bmatrix}$. Tada izrazi $Av_1=\lambda_1 v_1$ i $Av_2=\lambda_2 v_2$, zapisani zajedno, postaju jedna matrična jednadžba:

$$AP = P\Lambda$$

Budući da je matrica $A$ simetrična, njezini svojstveni vektori mogu se odabrati tako da budu *ortonormirani*, što matricu $P$ čini **ortogonalnom**. Njezin inverz tada je samo transponirana matrica: $P^{-1} = P^\top$. To nam omogućuje izraziti samu matricu $A$:

$$A = P\Lambda P^\top$$

To je **dijagonalizacija** matrice $A$. Geometrijski se sastoji od triju koraka: $P^\top$ *zakreće* koordinate u sustav svojstvenih vektora, $\Lambda$ neovisno *skalira* duž tih smjerova, koji su sada poravnati s koordinatnim osima, a $P$ *zakreće natrag*. U tom zakrenutom sustavu matrica $A$ je dijagonalna &mdash; izvandijagonalni element, koji povezuje $x$ i $y$, jednak je nuli. Kada je $A$ matrica kovarijance, taj izvandijagonalni element je $\mu_{11}$: dijagonalizacija matrice kovarijance i pronalaženje prirodnih osi elipse isti su postupak &mdash; oba znače prijelaz u koordinatni sustav u kojem raspršenost područja u smjerovima $x$ i $y$ više nije međusobno povezana.


In [ ]:
P = eigvecs
Lam = np.diag(eigvals)

print('A @ P:\n', np.round(A @ P, 4))
print('P @ Lambda:\n', np.round(P @ Lam, 4))
print('max |A@P - P@Lambda| (should be ~0):', np.abs(A @ P - P @ Lam).max())
print()

reconstructed = P @ Lam @ P.T
print('P @ Lambda @ P.T (reconstructed A):\n', np.round(reconstructed, 4))
print('original A:\n', A)
print('max |reconstructed - A|:', np.abs(reconstructed - A).max())
print()

diagonalized = P.T @ A @ P  # A, read in the eigenvector frame
print('P.T @ A @ P (A in its own eigenbasis -- should be diagonal):\n', np.round(diagonalized, 4))

## Od središnjih momenata do matrice kovarijance

Promatrajmo piksele područja kao uzorke iz dvodimenzionalne distribucije. Njegova matrica kovarijance, izražena pomoću središnjih momenata $\mu_{ij}$ i površine $\mu_{00}=m_{00}$, jest

$$A = \frac{1}{m_{00}}\begin{bmatrix}\mu_{20} & \mu_{11} \\ \mu_{11} & \mu_{02}\end{bmatrix}$$

To je ista formula koja se koristi za matricu kovarijance skupa točaka $(x,y)$, samo s ponderiranjem prema pripadnosti piksela području umjesto prema indeksu uzorka. Izvandijagonalni član $\mu_{11}$ odražava povezanost koordinata $x$ i $y$, pa je upravo to matrica koju želimo **dijagonalizirati**: njezini svojstveni vektori slijedit će glavnu i sporednu os područja &mdash; smjerove najveće i najmanje raspršenosti &mdash; a svojstvene vrijednosti mjerit će raspršenost duž svake osi. Isti postupak, primijenjen na matricu kovarijance općih podataka, koji ne moraju biti dvodimenzionalni pikseli, naziva se **analiza glavnih komponenti (PCA)**. Svojstveni vektori poredani prema svojstvenim vrijednostima predstavljaju *glavne komponente*, odnosno smjerove najveće varijance u podacima, te se često koriste za svođenje skupa podataka velike dimenzionalnosti na nekoliko najinformativnijih smjerova.


In [ ]:
def covariance_from_moments(binary):
    m = cv2.moments(binary, binaryImage=True)
    cx, cy = m['m10'] / m['m00'], m['m01'] / m['m00']
    cov = np.array([[m['mu20'], m['mu11']],
                     [m['mu11'], m['mu02']]]) / m['m00']
    return cov, (cx, cy)

## Dekompozicija na svojstvene vrijednosti i vektore daje orijentaciju i veličinu

Funkcija `np.linalg.eigh` vraća svojstvene vrijednosti uzlaznim redoslijedom te njihove svojstvene vektore kao stupce. Za ispunjenu elipsu s poluosima $a \ge b$, svojstvene vrijednosti su $\lambda_{\max} = a^2/4$ i $\lambda_{\min}=b^2/4$, pa vrijedi

$$a = 2\sqrt{\lambda_{\max}}, \qquad b = 2\sqrt{\lambda_{\min}}$$

Odgovarajući svojstveni vektori slijede glavnu i sporednu os. Primijetite da je kut glavne osi isti kao kut izračunat u Lekciji 5.


In [ ]:
def principal_axes(binary):
    cov, center = covariance_from_moments(binary)
    eigvals, eigvecs = np.linalg.eigh(cov)  # ascending order
    semi_axes = 2 * np.sqrt(np.clip(eigvals, 0, None))
    # reorder so index 0 is major (largest), index 1 is minor
    order = [1, 0]
    # eigvecs[:, order] has the two eigenvectors as its COLUMNS; transpose so that
    # unpacking it below (which iterates over ROWS) hands back the two actual eigenvectors,
    # not a scrambled mix of their x- and y-components
    return center, semi_axes[order], eigvecs[:, order].T


def draw_axes(ax, binary, color='red'):
    center, (a, b), (v_major, v_minor) = principal_axes(binary)
    cx, cy = center
    ax.imshow(binary, cmap='gray')
    for length, vec, lw in [(a, v_major, 2.5), (b, v_minor, 1.5)]:
        dx, dy = length * vec
        ax.plot([cx - dx, cx + dx], [cy - dy, cy + dy], c=color, linewidth=lw)
    ax.scatter(cx, cy, c=color, marker='x', s=60)
    ax.axis('off')
    return center, (a, b)

### Provjera na elipsi poznatih parametara

Kao u Lekciji 5, crtamo elipsu poznatih parametara i provjeravamo dobiva li procjena temeljena na svojstvenim vrijednostima i vektorima te parametre &mdash; ali ovaj put prikazujemo *obje* osi, a ne samo glavnu.


In [ ]:
binary = np.zeros((200, 200), dtype=np.uint8)
cv2.ellipse(binary, (100, 100), (70, 25), 30, 0, 360, 255, -1)

fig, ax = plt.subplots(figsize=(4, 4))
center, (a, b) = draw_axes(ax, binary)
ax.set_title('Major (thick) and minor (thin) axes from eigenvectors')
plt.show()

print(f'recovered semi-axes: a={a:.1f}, b={b:.1f}  (drawn with 70, 25)')

## Postupak radi i za proizvoljne oblike

Prednost ovog pristupa je u tome što područje uopće ne mora biti elipsa. Svaki binarni oblik ima *neku* ekvivalentnu elipsu &mdash; onu koja odgovara njegovoj površini, težištu i raspršenosti prema momentima drugog reda. Time dobivamo sažet opis područja proizvoljnog oblika pomoću 5 brojeva: dviju koordinata središta, duljina dviju poluosi i orijentacije.


In [ ]:
im_glasses = cv2.imread('../img/glasses_outline.png', cv2.IMREAD_GRAYSCALE)

shapes = {'Glasses': im_glasses}

fig, ax = plt.subplots(figsize=(5, 4))
center, (a, b) = draw_axes(ax, im_glasses)
ax.set_title(f'Glasses with axes\na={a:.0f}, b={b:.0f}', fontsize=10)
plt.show()

<p class="image-attribution">Izvor slike: Stan Birchfield</p>


Primijetite da ekvivalentna elipsa ne pokušava pratiti rub oblika &mdash; ona sažima *raspodjelu mase* oko težišta. Zbog toga presijeca uski most između dviju leća umjesto da ga prati, a njezin obris izlazi izvan leća pri vrhu i dnu, gdje se oblik naočala sužava.


## Ekscentricitet: koliko je oblik izdužen?

Omjer svojstvenih vrijednosti, odnosno poluosi, daje jedan broj koji opisuje izduženost, neovisno o orijentaciji i ukupnoj veličini:

$$\text{eccentricity} = \sqrt{1 - \frac{\lambda_{\min}}{\lambda_{\max}}}$$

Vrijednost se kreće od 0 (kružnica, obje su osi jednake) do gotovo 1 (vrlo tanak, izdužen oblik).


In [ ]:
print(f'{"shape":>10} {"a":>6} {"b":>6} {"eccentricity":>13}')
for name, img in shapes.items():
    _, (a, b), _ = principal_axes(img)
    ecc = np.sqrt(1 - (b / a) ** 2)
    print(f'{name:>10} {a:6.1f} {b:6.1f} {ecc:13.3f}')

### Zadaci

1. Nacrtajte savršenu kružnicu i provjerite je li njezin ekscentricitet jednak ili blizak 0.
2. Preko slike nacrtajte *stvarni* obris dobivene elipse, a ne samo njezine osi, pomoću `cv2.ellipse`, koristeći središte, `(2a, 2b)` kao pune duljine osi te kut orijentacije dobiven iz svojstvenih vektora. Usporedite ga s rezultatom funkcije `cv2.fitEllipse` primijenjene na konturu oblika &mdash; podudaraju li se?
3. Pokrenite `principal_axes` na ispunjenom području slike `glasses` nakon popunjavanja područja. Koliko se mijenjaju `a`, `b` i ekscentricitet? Odgovara li to vašoj intuitivnoj predodžbi o tome kako momenti ovise o *položaju* mase, a ne samo o ukupnoj silueti?
